In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

DATA_ROOT    = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\raw\LA\LA'
PROTOCOL_DIR = f'{DATA_ROOT}\\ASVspoof2019_LA_cm_protocols'
TRAIN_AUDIO  = f'{DATA_ROOT}\\ASVspoof2019_LA_train\\flac'
DEV_AUDIO    = f'{DATA_ROOT}\\ASVspoof2019_LA_dev\\flac'
EVAL_AUDIO   = f'{DATA_ROOT}\\ASVspoof2019_LA_eval\\flac'
SAVE_DIR     = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\processed'

os.makedirs(SAVE_DIR, exist_ok=True)

train_df = pd.read_csv(f'{PROTOCOL_DIR}\\ASVspoof2019.LA.cm.train.trn.txt',
    sep=' ', header=None,
    names=['speaker_id', 'file_id', 'env', 'attack_id', 'label'])

dev_df = pd.read_csv(f'{PROTOCOL_DIR}\\ASVspoof2019.LA.cm.dev.trl.txt',
    sep=' ', header=None,
    names=['speaker_id', 'file_id', 'env', 'attack_id', 'label'])

eval_df = pd.read_csv(f'{PROTOCOL_DIR}\\ASVspoof2019.LA.cm.eval.trl.txt',
    sep=' ', header=None,
    names=['speaker_id', 'file_id', 'env', 'attack_id', 'label'])

print("Paths ready ✅")
print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Eval: {len(eval_df)}")

In [ ]:
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000)
        
        # 1. MFCC — 40 features
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfcc_mean = mfcc.mean(axis=1)
        
        # 2. Spectral rolloff — 1 feature
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        rolloff_mean = rolloff.mean()
        
        # 3. Zero crossing rate — 1 feature
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = zcr.mean()
        
        # 4. Chroma — 12 features
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = chroma.mean(axis=1)
        
        # 5. Spectral contrast — 7 features
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = contrast.mean(axis=1)
        
        # total 61 features
        features = np.concatenate([
            mfcc_mean,
            [rolloff_mean],
            [zcr_mean],
            chroma_mean,
            contrast_mean
        ])
        return features
    
    except Exception as e:
        return None

# ek file pe test
sample = f'{TRAIN_AUDIO}\\{train_df.iloc[0]["file_id"]}.flac'
feat = extract_features(sample)
print(f"Feature vector length: {len(feat)}")
print(f"First 5 values: {feat[:5].round(3)}")

In [ ]:
def extract_all(df, audio_dir, split_name):
    X, y_labels = [], []
    failed = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=split_name):
        file_path = f'{audio_dir}\\{row["file_id"]}.flac'
        feat = extract_features(file_path)
        
        if feat is not None:
            X.append(feat)
            y_labels.append(1 if row['label'] == 'spoof' else 0)
        else:
            failed += 1
    
    X = np.array(X)
    y_labels = np.array(y_labels)
    print(f"{split_name} — Shape: {X.shape}, Failed: {failed}")
    return X, y_labels

# Train — ~21 minutes
X_train, y_train = extract_all(train_df, TRAIN_AUDIO, 'TRAIN')
np.save(f'{SAVE_DIR}\\X_train.npy', X_train)
np.save(f'{SAVE_DIR}\\y_train.npy', y_train)
print("Train saved ✅")

In [ ]:
# Dev — ~21 minutes
X_dev, y_dev = extract_all(dev_df, DEV_AUDIO, 'DEV')
np.save(f'{SAVE_DIR}\\X_dev.npy', X_dev)
np.save(f'{SAVE_DIR}\\y_dev.npy', y_dev)
print("Dev saved ✅")

# Eval — ~60 minutes
X_eval, y_eval = extract_all(eval_df, EVAL_AUDIO, 'EVAL')
np.save(f'{SAVE_DIR}\\X_eval.npy', X_eval)
np.save(f'{SAVE_DIR}\\y_eval.npy', y_eval)
print("Eval saved ✅")